# Advanced SQL Practice: E-Commerce Analytics

## Business Use Case: TechMart Online Store

**TechMart** is an online electronics retailer operating across Europe. The company needs to:

- **Track customer behavior** across cities to optimize marketing campaigns
- **Analyze product performance** by category to manage inventory
- **Monitor order trends** to forecast demand and staffing
- **Ensure data integrity** with automated stock management
- **Build KPI dashboards** for real-time business insights

### Data Model (ERD)

![Demo Schema ERD](docs/demo-schema-erd.png)

| Table | Description |
|-------|-------------|
| **customers** | Customer profiles with location data |
| **products** | Product catalog with pricing and inventory |
| **orders** | Transaction records linking customers to products |

---

## Setup: Database Connection

In [1]:
# Load environment and connect to PostgreSQL
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)

import os
from urllib.parse import quote_plus
from dotenv import load_dotenv

load_dotenv()

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = quote_plus(os.getenv('DB_PASSWORD'))
DB_NAME = os.getenv('DB_NAME')

os.environ['DATABASE_URL'] = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
print(f'Connecting to: {DB_HOST}:{DB_PORT}/{DB_NAME}')

Connecting to: postgres-rds-stack-postgres.ch8isoyag12f.eu-west-1.rds.amazonaws.com:5432/mydb


In [2]:
# Load SQL magic
%reload_ext sql
%config SqlMagic.displaylimit = 50

Tip: You may define configurations in /home/jupyter/notebooks/pyproject.toml or /home/jupyter/.jupysql/config.

Did not find user configurations in /home/jupyter/notebooks/pyproject.toml.

In [3]:
# Connect to PostgreSQL
from sqlalchemy import create_engine
engine = create_engine(os.environ["DATABASE_URL"])
%sql engine --alias postgres

---

## Part 1: Schema Creation

Create the TechMart database schema with proper constraints.

In [4]:
%%sql
-- Drop existing tables (if any) to start fresh
DROP TABLE IF EXISTS orders CASCADE;
DROP TABLE IF EXISTS products CASCADE;
DROP TABLE IF EXISTS customers CASCADE;

Running query in 'postgres'

++
||
++
++

In [5]:
%%sql
-- Create customers table
CREATE TABLE customers (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(150) UNIQUE NOT NULL,
    city VARCHAR(50) NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Create products table with inventory tracking
CREATE TABLE products (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    category VARCHAR(50) NOT NULL,
    stock INTEGER NOT NULL DEFAULT 0,
    -- Constraints for data integrity
    CONSTRAINT positive_price CHECK (price > 0),
    CONSTRAINT non_negative_stock CHECK (stock >= 0)
);

-- Create orders table with foreign keys
CREATE TABLE orders (
    id SERIAL PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(id),
    product_id INTEGER NOT NULL REFERENCES products(id),
    quantity INTEGER NOT NULL,
    order_date DATE NOT NULL DEFAULT CURRENT_DATE,
    status VARCHAR(20) NOT NULL DEFAULT 'pending',
    -- Constraints
    CONSTRAINT positive_quantity CHECK (quantity > 0 AND quantity <= 100),
    CONSTRAINT valid_status CHECK (status IN ('pending', 'processing', 'shipped', 'completed', 'cancelled'))
);

Running query in 'postgres'

++
||
++
++

---

## Part 2: Seed Data

Insert realistic sample data for TechMart operations.

In [6]:
%%sql
-- Insert customers across European cities
INSERT INTO customers (name, email, city) VALUES
    ('Ana Garcia', 'ana.garcia@email.com', 'Barcelona'),
    ('John Smith', 'john.smith@email.com', 'London'),
    ('Marie Dupont', 'marie.dupont@email.com', 'Paris'),
    ('Hans Mueller', 'hans.mueller@email.com', 'Berlin'),
    ('Sofia Rossi', 'sofia.rossi@email.com', 'Milan'),
    ('Carlos Lopez', 'carlos.lopez@email.com', 'Madrid'),
    ('Emma Wilson', 'emma.wilson@email.com', 'London'),
    ('Pierre Martin', 'pierre.martin@email.com', 'Paris'),
    ('Laura Sanchez', 'laura.sanchez@email.com', 'Barcelona'),
    ('Marco Bianchi', 'marco.bianchi@email.com', 'Milan'),
    ('Inactive Customer', 'inactive@email.com', 'Amsterdam');

Running query in 'postgres'

11 rows affected.

++
||
++
++

In [7]:
%%sql
-- Insert products across categories
INSERT INTO products (name, price, category, stock) VALUES
    -- Electronics
    ('MacBook Pro 14"', 1999.99, 'Electronics', 25),
    ('iPhone 15 Pro', 1199.99, 'Electronics', 50),
    ('Samsung Galaxy S24', 899.99, 'Electronics', 40),
    ('Sony WH-1000XM5', 349.99, 'Electronics', 100),
    ('iPad Air', 599.99, 'Electronics', 35),
    -- Accessories
    ('Apple AirPods Pro', 249.99, 'Accessories', 150),
    ('Logitech MX Master 3', 99.99, 'Accessories', 200),
    ('Samsung USB-C Hub', 49.99, 'Accessories', 300),
    ('Anker Charger 65W', 39.99, 'Accessories', 250),
    -- Gaming
    ('PlayStation 5', 499.99, 'Gaming', 15),
    ('Xbox Series X', 499.99, 'Gaming', 20),
    ('Nintendo Switch OLED', 349.99, 'Gaming', 45),
    -- Discontinued (for testing)
    ('Old Product', 19.99, 'Discontinued', 0);

Running query in 'postgres'

13 rows affected.

++
||
++
++

In [8]:
%%sql
-- Insert orders with various statuses and dates
INSERT INTO orders (customer_id, product_id, quantity, order_date, status) VALUES
    -- Ana Garcia - frequent buyer
    (1, 1, 1, '2026-01-05', 'completed'),
    (1, 6, 2, '2026-01-10', 'completed'),
    (1, 4, 1, '2026-01-20', 'shipped'),
    -- John Smith
    (2, 2, 1, '2026-01-08', 'completed'),
    (2, 7, 1, '2026-01-15', 'completed'),
    -- Marie Dupont - high value customer
    (3, 1, 2, '2026-01-12', 'completed'),
    (3, 5, 1, '2026-01-18', 'completed'),
    (3, 6, 3, '2026-01-25', 'processing'),
    -- Hans Mueller
    (4, 10, 1, '2026-01-14', 'completed'),
    (4, 12, 1, '2026-01-22', 'shipped'),
    -- Sofia Rossi
    (5, 3, 1, '2026-01-16', 'completed'),
    (5, 8, 2, '2026-01-24', 'completed'),
    -- Carlos Lopez
    (6, 11, 1, '2026-01-19', 'completed'),
    -- Emma Wilson
    (7, 4, 1, '2026-01-21', 'completed'),
    (7, 9, 3, '2026-01-26', 'pending'),
    -- Pierre Martin
    (8, 2, 1, '2026-01-23', 'shipped'),
    -- Laura Sanchez
    (9, 5, 1, '2026-01-27', 'pending'),
    -- Marco Bianchi
    (10, 1, 1, '2026-01-28', 'processing'),
    (10, 6, 1, '2026-01-28', 'processing');
    -- Note: Customer 11 (Inactive) has no orders

Running query in 'postgres'

19 rows affected.

++
||
++
++

In [9]:
%%sql
-- Verify data loaded correctly
SELECT 'customers' AS table_name, COUNT(*) AS rows FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders;

Running query in 'postgres'

3 rows affected.

table_name,rows
customers,11
products,13
orders,19


---

## Part 3: JOIN Operations

### 3.1 INNER JOIN - Only Matching Records

**Business Question:** "Which customers bought which products, and for how much?"

In [10]:
%%sql
-- Orders with customer and product details
SELECT 
    c.name AS customer,
    c.city,
    p.name AS product,
    p.category,
    o.quantity,
    p.price,
    (o.quantity * p.price) AS total,
    o.status
FROM orders o
INNER JOIN customers c ON o.customer_id = c.id
INNER JOIN products p ON o.product_id = p.id
ORDER BY total DESC;

Running query in 'postgres'

19 rows affected.

customer,city,product,category,quantity,price,total,status
Marie Dupont,Paris,"MacBook Pro 14""",Electronics,2,1999.99,3999.98,completed
Marco Bianchi,Milan,"MacBook Pro 14""",Electronics,1,1999.99,1999.99,processing
Ana Garcia,Barcelona,"MacBook Pro 14""",Electronics,1,1999.99,1999.99,completed
John Smith,London,iPhone 15 Pro,Electronics,1,1199.99,1199.99,completed
Pierre Martin,Paris,iPhone 15 Pro,Electronics,1,1199.99,1199.99,shipped
Sofia Rossi,Milan,Samsung Galaxy S24,Electronics,1,899.99,899.99,completed
Marie Dupont,Paris,Apple AirPods Pro,Accessories,3,249.99,749.97,processing
Laura Sanchez,Barcelona,iPad Air,Electronics,1,599.99,599.99,pending
Marie Dupont,Paris,iPad Air,Electronics,1,599.99,599.99,completed
Hans Mueller,Berlin,PlayStation 5,Gaming,1,499.99,499.99,completed


### 3.2 LEFT JOIN - Include All from Left Table

**Business Question:** "Which customers have NEVER placed an order?" (Potential for marketing campaigns)

In [11]:
%%sql
-- Find customers without any orders
SELECT 
    c.name,
    c.email,
    c.city,
    c.created_at
FROM customers c
LEFT JOIN orders o ON c.id = o.customer_id
WHERE o.id IS NULL;

Running query in 'postgres'

1 rows affected.

name,email,city,created_at
Inactive Customer,inactive@email.com,Amsterdam,2026-01-29 11:13:17.200135


In [12]:
%%sql
-- Count orders per customer (including those with 0 orders)
SELECT 
    c.name,
    c.city,
    COUNT(o.id) AS order_count,
    COALESCE(SUM(o.quantity), 0) AS total_items
FROM customers c
LEFT JOIN orders o ON c.id = o.customer_id
GROUP BY c.id, c.name, c.city
ORDER BY order_count DESC;

Running query in 'postgres'

11 rows affected.

name,city,order_count,total_items
Marie Dupont,Paris,3,6
Ana Garcia,Barcelona,3,4
Sofia Rossi,Milan,2,3
John Smith,London,2,2
Hans Mueller,Berlin,2,2
Marco Bianchi,Milan,2,2
Emma Wilson,London,2,4
Pierre Martin,Paris,1,1
Carlos Lopez,Madrid,1,1
Laura Sanchez,Barcelona,1,1


### 3.3 Products Never Ordered

**Business Question:** "Which products have never been sold?" (Inventory optimization)

In [13]:
%%sql
-- Products that have never been ordered
SELECT 
    p.name,
    p.category,
    p.price,
    p.stock
FROM products p
LEFT JOIN orders o ON p.id = o.product_id
WHERE o.id IS NULL;

Running query in 'postgres'

1 rows affected.

name,category,price,stock
Old Product,Discontinued,19.99,0


### Exercise: JOINs

1. List all orders from London customers with product details
2. Find customers who ordered Electronics products
3. Show products that have been ordered more than once

In [14]:
%%sql
-- Your solution here (remove the placeholder line below)
SELECT 1 AS placeholder;

Running query in 'postgres'

1 rows affected.

placeholder
1


---

## Part 4: Aggregations & Grouping

### 4.1 Basic Aggregations

**Business Question:** "What are our overall sales metrics?"

In [15]:
%%sql
-- Overall business metrics
SELECT 
    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(quantity) AS total_items_sold,
    ROUND(AVG(quantity), 2) AS avg_items_per_order
FROM orders
WHERE status = 'completed';

Running query in 'postgres'

1 rows affected.

total_orders,unique_customers,total_items_sold,avg_items_per_order
11,7,14,1.27


### 4.2 GROUP BY - Aggregation by Category

**Business Question:** "What is our revenue by product category?"

In [16]:
%%sql
-- Sales by product category
SELECT 
    p.category,
    COUNT(o.id) AS orders,
    SUM(o.quantity) AS items_sold,
    ROUND(SUM(o.quantity * p.price)::numeric, 2) AS revenue,
    ROUND(AVG(p.price)::numeric, 2) AS avg_price
FROM orders o
JOIN products p ON o.product_id = p.id
WHERE o.status = 'completed'
GROUP BY p.category
ORDER BY revenue DESC;

Running query in 'postgres'

3 rows affected.

category,orders,items_sold,revenue,avg_price
Electronics,6,7,9049.93,1174.99
Gaming,2,2,999.98,499.99
Accessories,3,5,699.95,133.32


In [17]:
%%sql
-- Sales by city
SELECT 
    c.city,
    COUNT(DISTINCT c.id) AS customers,
    COUNT(o.id) AS orders,
    ROUND(SUM(o.quantity * p.price)::numeric, 2) AS revenue
FROM customers c
JOIN orders o ON c.id = o.customer_id
JOIN products p ON o.product_id = p.id
GROUP BY c.city
ORDER BY revenue DESC;

Running query in 'postgres'

6 rows affected.

city,customers,orders,revenue
Paris,2,4,6549.93
Barcelona,2,4,3449.95
Milan,2,4,3249.95
London,2,4,1769.94
Berlin,1,2,849.98
Madrid,1,1,499.99


### 4.3 HAVING - Filter Aggregated Results

**Business Question:** "Who are our VIP customers (spent more than $1000)?"

In [18]:
%%sql
-- VIP customers (total spending > $1000)
SELECT 
    c.name,
    c.city,
    COUNT(o.id) AS total_orders,
    ROUND(SUM(o.quantity * p.price)::numeric, 2) AS total_spent
FROM customers c
JOIN orders o ON c.id = o.customer_id
JOIN products p ON o.product_id = p.id
WHERE o.status = 'completed'
GROUP BY c.id, c.name, c.city
HAVING SUM(o.quantity * p.price) > 1000
ORDER BY total_spent DESC;

Running query in 'postgres'

3 rows affected.

name,city,total_orders,total_spent
Marie Dupont,Paris,2,4599.97
Ana Garcia,Barcelona,2,2499.97
John Smith,London,2,1299.98


### Exercise: Aggregations

1. Find the best-selling product (by quantity)
2. Calculate average order value by status
3. Find cities with more than 2 customers

In [19]:
%%sql
-- Your solution here (remove the placeholder line below)
SELECT 1 AS placeholder;

Running query in 'postgres'

1 rows affected.

placeholder
1


---

## Part 5: Subqueries & CTEs

### 5.1 Subqueries

**Business Question:** "Which customers bought our most expensive product?"

In [20]:
%%sql
-- Customers who bought the most expensive product
SELECT DISTINCT c.name, c.email, c.city
FROM customers c
WHERE c.id IN (
    SELECT o.customer_id 
    FROM orders o
    WHERE o.product_id = (
        SELECT id FROM products 
        ORDER BY price DESC 
        LIMIT 1
    )
);

Running query in 'postgres'

3 rows affected.

name,email,city
Ana Garcia,ana.garcia@email.com,Barcelona
Marco Bianchi,marco.bianchi@email.com,Milan
Marie Dupont,marie.dupont@email.com,Paris


In [21]:
%%sql
-- Products priced above average
SELECT name, category, price
FROM products
WHERE price > (SELECT AVG(price) FROM products)
ORDER BY price DESC;

Running query in 'postgres'

4 rows affected.

name,category,price
"MacBook Pro 14""",Electronics,1999.99
iPhone 15 Pro,Electronics,1199.99
Samsung Galaxy S24,Electronics,899.99
iPad Air,Electronics,599.99


### 5.2 CTEs (Common Table Expressions)

CTEs make complex queries more readable. Same logic, better structure.

In [22]:
%%sql
-- CTE: Customer spending analysis
WITH customer_spending AS (
    SELECT 
        c.id,
        c.name,
        c.city,
        SUM(o.quantity * p.price) AS total_spent
    FROM customers c
    JOIN orders o ON c.id = o.customer_id
    JOIN products p ON o.product_id = p.id
    WHERE o.status = 'completed'
    GROUP BY c.id, c.name, c.city
),
spending_stats AS (
    SELECT 
        AVG(total_spent) AS avg_spending,
        MAX(total_spent) AS max_spending
    FROM customer_spending
)
SELECT 
    cs.name,
    cs.city,
    ROUND(cs.total_spent::numeric, 2) AS total_spent,
    CASE 
        WHEN cs.total_spent > ss.avg_spending THEN 'Above Average'
        ELSE 'Below Average'
    END AS spending_tier
FROM customer_spending cs, spending_stats ss
ORDER BY cs.total_spent DESC;

Running query in 'postgres'

7 rows affected.

name,city,total_spent,spending_tier
Marie Dupont,Paris,4599.97,Above Average
Ana Garcia,Barcelona,2499.97,Above Average
John Smith,London,1299.98,Below Average
Sofia Rossi,Milan,999.97,Below Average
Hans Mueller,Berlin,499.99,Below Average
Carlos Lopez,Madrid,499.99,Below Average
Emma Wilson,London,349.99,Below Average


### 5.3 Window Functions

**Business Question:** "Rank our customers by spending"

In [23]:
%%sql
-- Rank customers by total spending
SELECT 
    c.name,
    c.city,
    ROUND(SUM(o.quantity * p.price)::numeric, 2) AS total_spent,
    RANK() OVER (ORDER BY SUM(o.quantity * p.price) DESC) AS spending_rank
FROM customers c
JOIN orders o ON c.id = o.customer_id
JOIN products p ON o.product_id = p.id
GROUP BY c.id, c.name, c.city;

Running query in 'postgres'

10 rows affected.

name,city,total_spent,spending_rank
Marie Dupont,Paris,5349.94,1
Ana Garcia,Barcelona,2849.96,2
Marco Bianchi,Milan,2249.98,3
John Smith,London,1299.98,4
Pierre Martin,Paris,1199.99,5
Sofia Rossi,Milan,999.97,6
Hans Mueller,Berlin,849.98,7
Laura Sanchez,Barcelona,599.99,8
Carlos Lopez,Madrid,499.99,9
Emma Wilson,London,469.96,10


In [24]:
%%sql
-- Running total of orders by date
SELECT 
    order_date,
    COUNT(*) AS daily_orders,
    SUM(COUNT(*)) OVER (ORDER BY order_date) AS cumulative_orders
FROM orders
GROUP BY order_date
ORDER BY order_date;

Running query in 'postgres'

18 rows affected.

order_date,daily_orders,cumulative_orders
2026-01-05,1,1
2026-01-08,1,2
2026-01-10,1,3
2026-01-12,1,4
2026-01-14,1,5
2026-01-15,1,6
2026-01-16,1,7
2026-01-18,1,8
2026-01-19,1,9
2026-01-20,1,10


### Exercise: Subqueries & CTEs

1. Using a CTE, find products that have never been ordered
2. Create a query that shows each product's sales compared to category average
3. Use a window function to show running revenue by date

In [25]:
%%sql
-- Your solution here (remove the placeholder line below)
SELECT 1 AS placeholder;

Running query in 'postgres'

1 rows affected.

placeholder
1


---

## Part 6: Triggers & Automation

### 6.1 Stock Update Trigger

**Business Need:** Automatically decrease product stock when an order is placed.

In [26]:
%%sql
-- Drop existing trigger and function if they exist
DROP TRIGGER IF EXISTS trigger_update_stock ON orders;
DROP FUNCTION IF EXISTS update_stock();

Running query in 'postgres'

++
||
++
++

In [27]:
%%sql
-- Create function to update stock
CREATE OR REPLACE FUNCTION update_stock()
RETURNS TRIGGER AS $$
BEGIN
    -- Decrease stock when order is placed
    UPDATE products
    SET stock = stock - NEW.quantity
    WHERE id = NEW.product_id;
    
    -- Check if stock went negative
    IF (SELECT stock FROM products WHERE id = NEW.product_id) < 0 THEN
        RAISE EXCEPTION 'Insufficient stock for product ID %', NEW.product_id;
    END IF;
    
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

Running query in 'postgres'

++
||
++
++

In [28]:
%%sql
-- Attach trigger to orders table
CREATE TRIGGER trigger_update_stock
AFTER INSERT ON orders
FOR EACH ROW
EXECUTE FUNCTION update_stock();

Running query in 'postgres'

++
||
++
++

In [29]:
%%sql
-- Check current stock for AirPods
SELECT id, name, stock FROM products WHERE name LIKE '%AirPods%';

Running query in 'postgres'

1 rows affected.

id,name,stock
6,Apple AirPods Pro,150


In [30]:
%%sql
-- Test the trigger: Place an order
INSERT INTO orders (customer_id, product_id, quantity, status)
VALUES (1, 6, 5, 'pending');  -- Order 5 AirPods

Running query in 'postgres'

1 rows affected.

++
||
++
++

In [31]:
%%sql
-- Verify stock was automatically decreased
SELECT id, name, stock FROM products WHERE name LIKE '%AirPods%';

Running query in 'postgres'

1 rows affected.

id,name,stock
6,Apple AirPods Pro,145


### 6.2 Views for KPI Dashboards

In [32]:
%%sql
-- Drop view if exists
DROP VIEW IF EXISTS customer_summary;

Running query in 'postgres'

++
||
++
++

In [33]:
%%sql
-- Create customer summary view
CREATE VIEW customer_summary AS
SELECT 
    c.id,
    c.name,
    c.city,
    COUNT(o.id) AS total_orders,
    COALESCE(ROUND(SUM(o.quantity * p.price)::numeric, 2), 0) AS total_revenue,
    MAX(o.order_date) AS last_order_date
FROM customers c
LEFT JOIN orders o ON c.id = o.customer_id
LEFT JOIN products p ON o.product_id = p.id
GROUP BY c.id, c.name, c.city;

Running query in 'postgres'

++
||
++
++

In [34]:
%%sql
-- Use the view like a table
SELECT * FROM customer_summary 
WHERE total_revenue > 500
ORDER BY total_revenue DESC;

Running query in 'postgres'

8 rows affected.

id,name,city,total_orders,total_revenue,last_order_date
3,Marie Dupont,Paris,3,5349.94,2026-01-25
1,Ana Garcia,Barcelona,4,4099.91,2026-01-29
10,Marco Bianchi,Milan,2,2249.98,2026-01-28
2,John Smith,London,2,1299.98,2026-01-15
8,Pierre Martin,Paris,1,1199.99,2026-01-23
5,Sofia Rossi,Milan,2,999.97,2026-01-24
4,Hans Mueller,Berlin,2,849.98,2026-01-22
9,Laura Sanchez,Barcelona,1,599.99,2026-01-27


### Exercise: Triggers

1. Create an audit log table and trigger that logs every order insertion
2. Create a trigger that prevents orders for products with 0 stock
3. Create a daily_sales view that shows orders, revenue, and items by date

In [35]:
%%sql
-- Your solution here (remove the placeholder line below)
SELECT 1 AS placeholder;

Running query in 'postgres'

1 rows affected.

placeholder
1


---

## Part 7: Constraints & Assertions

### 7.1 CHECK Constraints (Already Applied)

Our schema already includes these constraints:

- `positive_price` - Products must have price > 0
- `non_negative_stock` - Stock cannot be negative
- `positive_quantity` - Order quantity must be 1-100
- `valid_status` - Status must be one of the allowed values

In [36]:
%%sql
-- Test constraint: Try to insert negative price (will fail)
INSERT INTO products (name, price, category, stock)
VALUES ('Bad Product', -10.00, 'Test', 100);

Running query in 'postgres'

RuntimeError: (psycopg2.errors.CheckViolation) new row for relation "products" violates check constraint "positive_price"
DETAIL:  Failing row contains (14, Bad Product, -10.00, Test, 100).

[SQL: INSERT INTO products (name, price, category, stock)
VALUES ('Bad Product', -10.00, 'Test', 100);]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [37]:
%%sql
-- Test constraint: Try invalid status (will fail)
INSERT INTO orders (customer_id, product_id, quantity, status)
VALUES (1, 1, 1, 'invalid_status');

Running query in 'postgres'

RuntimeError: (psycopg2.errors.CheckViolation) new row for relation "orders" violates check constraint "valid_status"
DETAIL:  Failing row contains (21, 1, 1, 1, 2026-01-29, invalid_status).

[SQL: INSERT INTO orders (customer_id, product_id, quantity, status)
VALUES (1, 1, 1, 'invalid_status');]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


### 7.2 Adding New Constraints

In [38]:
%%sql
-- Add constraint: Email must contain @
ALTER TABLE customers
ADD CONSTRAINT valid_email CHECK (email LIKE '%@%.%');

Running query in 'postgres'

++
||
++
++

In [39]:
%%sql
-- Add constraint: Category must be from allowed list
ALTER TABLE products
ADD CONSTRAINT valid_category 
CHECK (category IN ('Electronics', 'Accessories', 'Gaming', 'Discontinued'));

Running query in 'postgres'

++
||
++
++

### 7.3 Cross-Table Validation (Using Triggers)

PostgreSQL doesn't support SQL ASSERTION. Use triggers for cross-table rules.

In [40]:
%%sql
-- Create trigger to prevent ordering discontinued products
CREATE OR REPLACE FUNCTION check_product_available()
RETURNS TRIGGER AS $$
BEGIN
    IF (SELECT category FROM products WHERE id = NEW.product_id) = 'Discontinued' THEN
        RAISE EXCEPTION 'Cannot order discontinued product';
    END IF;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

DROP TRIGGER IF EXISTS trigger_check_product ON orders;
CREATE TRIGGER trigger_check_product
BEFORE INSERT ON orders
FOR EACH ROW
EXECUTE FUNCTION check_product_available();

Running query in 'postgres'

++
||
++
++

In [41]:
%%sql
-- Test: Try to order discontinued product (will fail)
INSERT INTO orders (customer_id, product_id, quantity, status)
VALUES (1, 13, 1, 'pending');  -- Product 13 is 'Old Product' (Discontinued)

Running query in 'postgres'

RuntimeError: (psycopg2.errors.RaiseException) Cannot order discontinued product
CONTEXT:  PL/pgSQL function check_product_available() line 4 at RAISE

[SQL: INSERT INTO orders (customer_id, product_id, quantity, status)
VALUES (1, 13, 1, 'pending');]
(Background on this error at: https://sqlalche.me/e/20/2j85)


### Exercise: Constraints

1. Add a constraint that prevents duplicate orders (same customer, product, date)
2. Create a trigger that ensures a customer can't have more than 10 pending orders
3. Add a CHECK constraint that price must be less than 10000

In [ ]:
%%sql
-- Your solution here (remove the placeholder line below)
SELECT 1 AS placeholder;

---

## Summary

### What We Learned

| Topic | Key Concepts |
|-------|-------------|
| **JOINs** | INNER, LEFT, RIGHT, FULL OUTER - combining tables |
| **Aggregations** | COUNT, SUM, AVG, MIN, MAX + GROUP BY + HAVING |
| **Subqueries** | Nested queries in SELECT, FROM, WHERE |
| **CTEs** | WITH clause for readable, reusable query blocks |
| **Window Functions** | RANK, ROW_NUMBER, running totals without collapsing rows |
| **Triggers** | Automated actions on INSERT/UPDATE/DELETE |
| **Views** | Saved queries as virtual tables |
| **Constraints** | CHECK, UNIQUE, FOREIGN KEY for data integrity |

### Business Value

- **JOINs** enable cross-table analytics (customer + orders + products)
- **Aggregations** power dashboards and KPIs
- **CTEs** make complex queries maintainable
- **Triggers** automate business rules (stock management)
- **Constraints** ensure data quality at the database level